# Experiment: VAE Exploration

Objective:
- Train the first-pass VAE on synthetic attack-profile corpora from inside the notebook.
- Inspect convergence, decoded samples, and run artifacts without leaving Jupyter.
- Keep the notebook aligned with the canonical runner in `experiments/train_vae.py`.


## Workflow

This notebook is intentionally split into two modes:
- **Run mode**: launch a training run by calling `run_from_args(...)` from `experiments.train_vae`.
- **Analysis mode**: point at an existing run directory and inspect its metrics, training history, and decoded samples.

Latest run when this notebook was scaffolded:
- `results/runs/vae/20260427_213933_default`


In [ ]:
from __future__ import annotations

import json
from argparse import Namespace
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from experiments.train_vae import run_from_args

PROJECT_ROOT = Path.cwd()
PROJECT_ROOT


In [ ]:
# Config: adjust here before training from the notebook
TRAIN_PATH = Path('data/attack_profiles/synthetic/train_random_v2.jsonl')
VALID_PATH = Path('data/attack_profiles/synthetic/valid_random_v2.jsonl')
RUN_NAME = 'notebook'
SEED = 1945
EPOCHS = 50
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
BETA = 0.05
LATENT_DIM = 4
HIDDEN_DIM = 32
DEVICE = 'auto'
NUM_WORKERS = 0
SAMPLE_COUNT = 16


## Train From Notebook

Run the next cell only when you want to launch a new training job from Jupyter.
It uses the same code path as the terminal script, so the artifacts are identical.


In [ ]:
# Uncomment to launch a fresh run from the notebook
# args = Namespace(
#     train_path=TRAIN_PATH,
#     valid_path=VALID_PATH,
#     output_root=Path('results/runs'),
#     run_name=RUN_NAME,
#     seed=SEED,
#     epochs=EPOCHS,
#     batch_size=BATCH_SIZE,
#     learning_rate=LEARNING_RATE,
#     beta=BETA,
#     latent_dim=LATENT_DIM,
#     hidden_dim=HIDDEN_DIM,
#     device=DEVICE,
#     num_workers=NUM_WORKERS,
#     sample_count=SAMPLE_COUNT,
# )
# run_dir = run_from_args(args, project_root=PROJECT_ROOT)
# print(run_dir)


## Analysis Setup

Point `RUN_DIR` at an existing VAE training run.
By default it uses the latest run that existed when this notebook was scaffolded.


In [ ]:
RUN_DIR = PROJECT_ROOT / 'results/runs/vae/20260427_213933_default'
METRICS_PATH = RUN_DIR / 'metrics_summary.json'
MANIFEST_PATH = RUN_DIR / 'run_manifest.json'
HISTORY_PATH = RUN_DIR / 'training_history.csv'
SAMPLES_PATH = RUN_DIR / 'sampled_profiles.json'

assert RUN_DIR.exists(), RUN_DIR
RUN_DIR


In [ ]:
metrics = json.loads(METRICS_PATH.read_text(encoding='utf-8'))
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
history = pd.read_csv(HISTORY_PATH)
samples = json.loads(SAMPLES_PATH.read_text(encoding='utf-8'))['profiles']

summary = {
    'train_samples': metrics['dataset']['train_samples'],
    'valid_samples': metrics['dataset']['valid_samples'],
    'latent_dim': metrics['model']['latent_dim'],
    'hidden_dim': metrics['model']['hidden_dim'],
    'parameter_count': metrics['model']['parameter_count'],
    'beta': metrics['model']['beta'],
    'best_epoch': metrics['training']['best_epoch'],
    'best_valid_loss': metrics['training']['best_valid_loss'],
    'final_valid_loss': metrics['training']['final_valid_loss'],
    'device': manifest['device'],
    'total_seconds': metrics['timing']['total_seconds'],
}
summary


In [ ]:
history.tail()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history.plot(x='epoch', y=['train_loss', 'valid_loss'], ax=axes[0], title='Total loss')
history.plot(x='epoch', y=['train_recon_loss', 'valid_recon_loss'], ax=axes[1], title='Reconstruction loss')
axes[0].grid(True, alpha=0.3)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
history.plot(x='epoch', y=['train_kl_loss', 'valid_kl_loss'], ax=ax, title='KL loss')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Decoded Sample Inspection

These are latent-prior samples decoded directly through the VAE and saved by the training script.
They are not yet rechecked by the feasibility/audit pipeline.


In [ ]:
samples_df = pd.DataFrame(samples)
samples_df[['profile_id', 'u_pos', 'base_bearing_rad', 'spread_rad', 'launch_delay_s', 'salvo_interval_s', 'u_boat_initial_speed_mps']].head(10)


In [ ]:
samples_df[['spread_rad', 'launch_delay_s', 'salvo_interval_s', 'u_boat_initial_speed_mps']].describe()


## Notes

Initial read on the scaffolded reference run:
- validation loss was still improving at epoch 50
- no obvious instability in KL or reconstruction terms
- decoded samples look numerically reasonable, but still need post-decode feasibility and audit gating


## Next Steps

- Add a notebook cell that loads `checkpoints/model_best.pt` and samples directly from the checkpoint.
- Add post-decode feasibility + audit filtering for VAE-generated profiles.
- Compare generated-sample distributions against the source dataset audit outputs.
